# Commute Network Personal Dataset + Related Dataset + Workflow

**Mapping Systems — Assignment**

This notebook builds a personal GeoJSON network describing my daily commute
between home (Hamilton Heights) and Columbia's Morningside campus, plus the
errands and relationships that structure that walk. It then sketches the
proposed workflow for relating this personal dataset to a public one: the
**MTA Subway Service Lines / Subway Stations** datasets (Open Data NY).

Sections:
1. Build the nodes (points)
2. Build the edges (lines)
3. Assemble and export `commute_network.geojson`
4. Visualize the network
5. Related dataset
6. Proposed workflow (spatial join + routing), as runnable code


## 1. Nodes

Each node is a place in the daily/weekly network, geocoded to real coordinates.

In [7]:
nodes = [
    {
        "id": "home",
        "name": "Home",
        "category": "residence",
        "note": "131st St & Amsterdam Ave — origin node of the daily network.",
        "lon": -73.9540369,
        "lat": 40.8163555,
    },
    {
        "id": "plowshares_coffee",
        "name": "Plowshares Coffee Roasters",
        "category": "coffee",
        "note": "Regular coffee stop on Amsterdam Ave, on the walk south from home.",
        "lon": -73.9556886,
        "lat": 40.8137198,
    },
    {
        "id": "nelson_cleaners",
        "name": "Nelson Cleaners",
        "category": "service",
        "note": "Laundry / tailoring, W 125th St.",
        "lon": -73.9559676,
        "lat": 40.8133193,
    },
    {
        "id": "subway_125_st",
        "name": "125 St Station (A/B/C/D)",
        "category": "transit",
        "note": "Typical subway entrance at St. Nicholas Ave & 125th St; used for the A train.",
        "lon": -73.9523400,
        "lat": 40.8111100,
    },
    {
        "id": "malvin_barbers",
        "name": "Malvin Barber Shop",
        "category": "service",
        "note": "Barber, Amsterdam Ave.",
        "lon": -73.9582542,
        "lat": 40.8112286,
    },
    {
        "id": "babas_marketplace",
        "name": "Baba's Marketplace",
        "category": "food_retail",
        "note": "24-hour deli / grocery, Amsterdam Ave.",
        "lon": -73.9583600,
        "lat": 40.8108715,
    },
    {
        "id": "trader_joes_125",
        "name": "Trader Joe's (125th St)",
        "category": "food_retail",
        "note": "Weekly grocery run, W 125th St.",
        "lon": -73.9469074,
        "lat": 40.8085527,
    },
    {
        "id": "schermerhorn_hall",
        "name": "Schermerhorn Hall",
        "category": "education_columbia",
        "note": "Typical entrance point onto Columbia's Morningside campus.",
        "lon": -73.9605576,
        "lat": 40.8086363,
    },
    {
        "id": "avery_hall",
        "name": "Avery Hall",
        "category": "education_columbia",
        "note": "MS. CDP (Computational Design Practices) studio.",
        "lon": -73.9609694,
        "lat": 40.8082793,
    },
    {
        "id": "fayerweather_hall",
        "name": "Fayerweather Hall",
        "category": "education_columbia",
        "note": "MS. HP (Historic Preservation) studio.",
        "lon": -73.9603477,
        "lat": 40.8082408,
    },
    {
        "id": "parsons_school",
        "name": "Parsons School of Design",
        "category": "education_affiliation",
        "note": "Undergraduate alma mater; occasional alumni visits, reached by subway.",
        "lon": -73.9944837,
        "lat": 40.7351297,
    },
    {
        "id": "partner_bushwick",
        "name": "Partner's residence (Broadway & Granite St)",
        "category": "personal",
        "note": "Bushwick, Brooklyn; reached via the A train plus transfer.",
        "lon": -73.9085358,
        "lat": 40.6818118,
    },
]

node_lookup = {n["id"]: (n["lon"], n["lat"]) for n in nodes}
print(f"{len(nodes)} nodes")

12 nodes


## 2. Edges

Each edge references node ids; coordinates are resolved from `node_lookup` so the geometry always stays in sync with the node table.

In [8]:
edges = [
    {
        "id": "daily_walk_corridor",
        "name": "Daily walk: Home → Campus (Amsterdam / St. Nicholas corridor)",
        "category": "route_walk",
        "mode": "walking",
        "frequency": "daily",
        "note": "Typical walking route threading the errand stops between home and the Schermerhorn Hall entrance to campus.",
        "path": [
            "home",
            "plowshares_coffee",
            "nelson_cleaners",
            "subway_125_st",
            "malvin_barbers",
            "babas_marketplace",
            "schermerhorn_hall",
        ],
    },
    {
        "id": "campus_studio_loop",
        "name": "Campus studio loop",
        "category": "route_walk",
        "mode": "walking",
        "frequency": "daily",
        "note": "Movement between the two degree studios and the campus entrance.",
        "path": ["schermerhorn_hall", "avery_hall", "fayerweather_hall"],
    },
    {
        "id": "grocery_spur",
        "name": "Grocery spur: 125 St Station → Trader Joe's",
        "category": "route_walk",
        "mode": "walking",
        "frequency": "weekly",
        "note": "Eastward spur along 125th St for the weekly grocery run.",
        "path": ["subway_125_st", "trader_joes_125"],
    },
    {
        "id": "subway_to_parsons",
        "name": "Subway desire line: 125 St → Parsons",
        "category": "route_subway",
        "mode": "subway (A train)",
        "frequency": "occasional",
        "note": "Straight desire line standing in for the actual A-train route to alumni visits at Parsons; not a literal track alignment.",
        "path": ["subway_125_st", "parsons_school"],
    },
    {
        "id": "subway_to_partner",
        "name": "Subway desire line: 125 St → Bushwick",
        "category": "route_subway",
        "mode": "subway (A train + transfer)",
        "frequency": "weekly",
        "note": "Straight desire line standing in for the A train plus a transfer (e.g. at Broadway Junction) to reach Bushwick; not a literal track alignment.",
        "path": ["subway_125_st", "partner_bushwick"],
    },
]

print(f"{len(edges)} edges")

5 edges


## 3. Assemble and export GeoJSON

In [9]:
import json

features = []

for n in nodes:
    features.append(
        {
            "type": "Feature",
            "properties": {
                "id": n["id"],
                "name": n["name"],
                "category": n["category"],
                "note": n["note"],
            },
            "geometry": {"type": "Point", "coordinates": [n["lon"], n["lat"]]},
        }
    )

for e in edges:
    coords = [list(node_lookup[node_id]) for node_id in e["path"]]
    features.append(
        {
            "type": "Feature",
            "properties": {k: v for k, v in e.items() if k != "path"},
            "geometry": {"type": "LineString", "coordinates": coords},
        }
    )

geojson = {
    "type": "FeatureCollection",
    "name": "khaled_commute_network",
    "description": (
        "A network dataset expressing the daily commute of a GSAPP graduate student "
        "between home in Hamilton Heights/Manhattanville and Columbia's Morningside campus, "
        "including the everyday service stops that structure the walk, and the longer "
        "subway-based connections that extend the network to a second campus affiliation "
        "and a partner's residence in Bushwick, Brooklyn."
    ),
    "features": features,
}

with open("commute_network.geojson", "w") as f:
    json.dump(geojson, f, indent=2)

print(
    f"Wrote commute_network.geojson with {len(features)} features "
    f"({len(nodes)} points, {len(edges)} lines)"
)

Wrote commute_network.geojson with 17 features (12 points, 5 lines)


## 4. Network Visualization

In [10]:
import plotly.graph_objects as go

MAPBOX_TOKEN = "pk.eyJ1Ijoia2hhbGVkYWxhbmplcnkiLCJhIjoiY21zOXJtdGE3MHM5NDJ3b2RhZDhlN3RzdiJ9.u-954HDL1cS8ZVODaKr3IQ"

node_colors = {
    "residence": "#1a1a1a",
    "coffee": "#8a5a2b",
    "service": "#4a6fa5",
    "transit": "#c94f4f",
    "food_retail": "#3a8a5a",
    "education_columbia": "#0057b7",
    "education_affiliation": "#7a4fc9",
    "personal": "#c9884f",
}

edge_colors = {
    "route_walk": "#888888",
    "route_subway": "#c94f4f",
}

node_lons = [n["lon"] for n in nodes]
node_lats = [n["lat"] for n in nodes]
node_names = [n["name"] for n in nodes]
node_categories = [n["category"] for n in nodes]
node_hover = [f"<b>{n['name']}</b><br>{n['note']}" for n in nodes]
node_marker_colors = [node_colors.get(cat, "#333333") for cat in node_categories]

fig = go.Figure()

for edge in edges:
    coords = [node_lookup[node_id] for node_id in edge["path"]]
    lons = [coord[0] for coord in coords]
    lats = [coord[1] for coord in coords]
    fig.add_trace(
        go.Scattermapbox(
            lon=lons,
            lat=lats,
            mode="lines",
            line=dict(width=3, color=edge_colors.get(edge["category"], "#999999")),
            hoverinfo="text",
            text=edge["name"],
            showlegend=False,
        )
    )

fig.add_trace(
    go.Scattermapbox(
        lon=node_lons,
        lat=node_lats,
        mode="markers+text",
        text=node_names,
        textposition="top center",
        hovertext=node_hover,
        hoverinfo="text",
        marker=dict(size=12, color=node_marker_colors),
        showlegend=False,
    )
)

fig.update_layout(
    title="Commute network — nodes and edges",
    mapbox=dict(
        style="mapbox://styles/mapbox/light-v10",
        center=dict(lat=40.80, lon=-73.96),
        zoom=11,
        accesstoken=MAPBOX_TOKEN,
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    width=1000,
    height=700,
)

fig.show()

/var/folders/_1/khzps_ks3rb9cj3hk5m1fvh80000gn/T/ipykernel_17797/2052746470.py:35: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(
/var/folders/_1/khzps_ks3rb9cj3hk5m1fvh80000gn/T/ipykernel_17797/2052746470.py:47: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(


## 5. Related dataset

**MTA Subway Service Lines** and **MTA Subway Stations**, published via Open Data NY
(data.ny.gov):

- Subway Service Lines (track geometry, LineString): https://data.ny.gov/d/s692-irgq
- MTA Subway Stations (station points, GTFS Stop IDs): https://data.ny.gov/Transportation/MTA-Subway-Stations/39hk-dx4f

Both are public, actively maintained, and downloadable directly as GeoJSON — no proxy
data required. The pairing is deliberate: this personal network already contains a
subway node (`subway_125_st`) and two subway "desire lines" (straight-line stand-ins
for the actual A-train route), so relating it to the MTA data lets those desire lines
be resolved into real, routable transit geometry.


## 6. Proposed workflow

**Step 1 — Spatial join.** Snap the `subway_125_st` node to its real-world counterpart
in the MTA Stations dataset (nearest-neighbor match within ~50m), pulling in its GTFS
Stop ID / Complex ID.

**Step 2 — Route the desire lines.** Build a graph from the Subway Service Lines
geometry and compute the shortest path between the matched station and the stations
nearest Parsons and Bushwick, replacing each straight desire line with the actual
routed track geometry.

**Step 3 — Output.** A combined network where the walking edges stay as drawn (they're
already accurate — I walk them), and the subway edges are now real transit paths,
taggable with stop-to-stop travel time from GTFS schedule data.

The two cells below are written to run against the actual downloaded MTA files —
they're guarded with `try/except` since `geopandas` isn't installed in this
environment. To run them for real: `pip install geopandas networkx`, download the two
datasets above as GeoJSON into this folder as `subway_lines.geojson` and
`subway_stations.geojson`, then re-run.


In [11]:
# Step 1: spatial join — snap subway_125_st to the nearest real MTA station
try:
    import os
    import geopandas as gpd
    from shapely.geometry import Point

    stations_path = "subway_stations.geojson"
    if not os.path.exists(stations_path):
        raise FileNotFoundError(f"{stations_path} not found")

    stations = gpd.read_file(stations_path).to_crs(epsg=2263)
    my_point = gpd.GeoSeries(
        [Point(node_lookup["subway_125_st"])], crs="EPSG:4326"
    ).to_crs(epsg=2263)

    stations["dist_ft"] = stations.geometry.distance(my_point.iloc[0])
    nearest = stations.sort_values("dist_ft").iloc[0]
    print(
        f"Nearest station: {nearest.get('STATION_NA', nearest.get('name', 'unknown'))} "
        f"({nearest['dist_ft']:.0f} ft away)"
    )

except (ImportError, FileNotFoundError) as e:
    print(f"[skipped locally: {type(e).__name__} — {e}]")
    print(
        "Run this cell after installing geopandas and downloading subway_stations.geojson."
    )

[skipped locally: FileNotFoundError — subway_stations.geojson not found]
Run this cell after installing geopandas and downloading subway_stations.geojson.


In [12]:
# Step 2: route the desire lines onto the real subway track graph
try:
    import os
    import geopandas as gpd
    import networkx as nx
    from shapely.geometry import Point

    lines_path = "subway_lines.geojson"
    if not os.path.exists(lines_path):
        raise FileNotFoundError(f"{lines_path} not found")

    lines = gpd.read_file(lines_path)

    G = nx.Graph()
    for _, row in lines.iterrows():
        coords = list(row.geometry.coords)
        for a, b in zip(coords[:-1], coords[1:]):
            G.add_edge(a, b, weight=Point(a).distance(Point(b)))

    # Example: nearest graph nodes to our 125 St and Parsons points
    def nearest_graph_node(pt, graph):
        return min(graph.nodes, key=lambda n: Point(n).distance(Point(pt)))

    start = nearest_graph_node(node_lookup["subway_125_st"], G)
    end = nearest_graph_node(node_lookup["parsons_school"], G)
    path = nx.shortest_path(G, start, end, weight="weight")
    print(f"Routed path from 125 St toward Parsons: {len(path)} track vertices")

except (ImportError, FileNotFoundError, NameError) as e:
    print(f"[skipped locally: {type(e).__name__} — {e}]")
    print(
        "Run this cell after installing geopandas/networkx and downloading subway_lines.geojson."
    )

[skipped locally: FileNotFoundError — subway_lines.geojson not found]
Run this cell after installing geopandas/networkx and downloading subway_lines.geojson.


## Reflection

The interesting contrast here is between the two edge types: the walking edges are
literal and felt — drawn from memory of the actual sidewalk path — while the subway
edges start as pure "desire lines," expressing *relationship* rather than geography,
and only become geographically real once joined to the MTA dataset. That gap between
felt proximity and routed distance (a five-minute walk to Fayerweather vs. an hour of
transit to Bushwick) is itself a way of visualizing how unevenly the city's transit
infrastructure compresses or stretches distance in a personal mental map.
